# Mean Variance Optimization with Scipy

In [3]:
import sys
sys.path.insert(0,'/Users/martincaroe/Desktop/MARTIN/Copenhagen Business School/Msc Economics and Business Administration - Finance and Investments /3 Semester/Python for the Financial Economics/Fork/python_for_the_financial_economist')


# import relevant packages
import numpy as np
import pandas as pd
from scipy import stats, optimize
import matplotlib.pyplot as plt

# own functions from codelib
from codelib.statistics import moments as mom
from codelib.portfolio_optimization.mean_variance import portfolio_mean, portfolio_std, portfolio_variance, minimum_variance_portfolio

from codelib.visualization.layout import DefaultStyle
DefaultStyle();

Consider the portfolio optimization problem 


where the expected return vector is given by 

$$
\boldsymbol{\mu} = \begin{pmatrix} 0.032 \\ 0.0322 \\ 0.084 \\ 0.082 \end{pmatrix},
$$

the vector of volatilites is given by 

$$
\mathbf{v} = \begin{pmatrix} 0.05 \\ 0.05 \\ 0.22 \\ 0.22 \end{pmatrix},
$$

and the correlation matrix is given by 

$$
\mathbf{C} = \begin{pmatrix} 1.0 & 0.85 & 0.5 & 0.45 \\
                      0.85 & 1.0 & 0.5 & 0.45 \\
                      0.5 & 0.5 & 1.0 & 0.9 \\
                      0.45 & 0.45 & 0.9 & 1.0 \end{pmatrix}
$$

In [4]:
corr_mat = np.array([[1.0, 0.85, 0.5, 0.45],
                     [0.85, 1.0, 0.5, 0.45],
                     [0.5, 0.5, 1.0, 0.9],
                     [0.45, 0.45, 0.9, 1.0]])

vols = np.array([5.0, 5.0, 22.0, 22.0]) / 100.0
mu = np.array([3.2, 3.22, 8.4, 8.2]) / 100.0

cov_mat = mom.corr_to_cov_matrix(corr_mat, vols)

## Problem 1

Set up the minimization problem that enable you to find the minimum variance portfolio using `scipy.optimize`. Apply a budget constraint requiring the portfolio weights to sum to one. Compare with the analytical solution. 

In [5]:
"""
Numerical solution with scipy
"""
def port_variance(w):
    return w @ cov_mat @ w

budget_constraint = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}   # sum(w) - 1 = 0

w0 = np.repeat(1.0 / 4.0, 4)          # start: equal weights

res = optimize.minimize(port_variance, w0, method='SLSQP',
                        constraints=[budget_constraint],
                        options={'ftol': 1e-12})    # <- tight tolerance, see note below
w_numerical = res.x

"""
Analytical solution (formula from Lecture 5)
"""
ones = np.ones(4)
cov_inv = np.linalg.inv(cov_mat)
w_analytical = cov_inv @ ones / (ones @ cov_inv @ ones)

print(w_numerical)
print(w_analytical)

[ 0.53892691  0.53892691 -0.1001686   0.02231478]
[ 0.53892691  0.53892691 -0.1001686   0.02231479]


**Tolkning:** The numerical optimizer and the analytical formula give the same answer, which confirms the setup is correct. The minimum-variance portfolio puts more than half of the money in each of the two calm assets (1 and 2) and actually SHORTS asset 3 (−10%), because asset 3 and 4 are very highly correlated (0.9), so shorting one against the other helps cancel risk. Its expected return is only about 2.8%, because minimizing risk alone pays no attention to return.

## Problem 2

Minimize the portfolio variance subject to the constraints that 

* The expected return should be above 6%
* No shorting (all weights should be equal or greater than 0)
* The portfolio weights need to sum to one (budget constraint)

In [6]:
"""
Constraints: budget, minimum expected return, no shorting
"""
budget_constraint = {'type': 'eq',   'fun': lambda w: np.sum(w) - 1.0}
return_constraint = {'type': 'ineq', 'fun': lambda w: w @ mu - 0.06}    # >= 0

bounds = [(0.0, None)] * 4          # each weight >= 0, no upper limit

res = optimize.minimize(port_variance, w0, method='SLSQP',
                        constraints=[budget_constraint, return_constraint],
                        bounds=bounds,
                        options={'ftol': 1e-12})

w_opt = res.x
print(w_opt)
print('Expected return:', w_opt @ mu)
print('Std:', np.sqrt(w_opt @ cov_mat @ w_opt))

[0.10444782 0.35063367 0.34197387 0.20294464]
Expected return: 0.060000000000000046
Std: 0.12967804947852918


**Tolkning:** To reach the required 6% return, the portfolio has to move a large share of money into the risky assets 3 and 4 (about 55% together, compared to essentially 0% before), and the risk rises from about 4.5% to about 13%. The return constraint is met exactly (6.0%), which shows it is the constraint that "bites": the unconstrained minimum-variance portfolio only earned 2.8%, so it was not allowed. Banning short-selling also removes the −10% position in asset 3 from Problem 1.